# Create Dataframes

Using the F1 source data files in the folder "f1-sourcefiles/" create DataFrames.



In [2]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

26/03/23 23:53:40 WARN Utils: Your hostname, DESKTOP-OQT8U26 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/23 23:53:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/robyip/projects/pyspark-deltalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/robyip/.ivy2/cache
The jars for the packages stored in: /home/robyip/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-824b3a78-4257-4c81-8b08-370622dc3576;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 161ms :: artifacts dl 6ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0 

In [3]:
# Load status.csv file
import time

start = time.time()

df = spark.read.csv("f1-sourcefiles/status.csv", header=True, inferSchema=True)

df.show()

end = time.time()

print(f"Time takes: {end - start:.2f} seconds")


+--------+------------+
|statusId|      status|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
|       6|     Gearbox|
|       7|Transmission|
|       8|      Clutch|
|       9|  Hydraulics|
|      10|  Electrical|
|      11|      +1 Lap|
|      12|     +2 Laps|
|      13|     +3 Laps|
|      14|     +4 Laps|
|      15|     +5 Laps|
|      16|     +6 Laps|
|      17|     +7 Laps|
|      18|     +8 Laps|
|      19|     +9 Laps|
|      20|    Spun off|
+--------+------------+
only showing top 20 rows

Time takes: 3.76 seconds


In [8]:
# Use commands after loading into a Dataframe
df.show()         # Preview data
df.printSchema()  # check column types
print("Row count: ", df.count())       # number of rows
print("Columnns: ", df.columns)       # list of column names

+--------+------------+
|statusId|      status|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
|       6|     Gearbox|
|       7|Transmission|
|       8|      Clutch|
|       9|  Hydraulics|
|      10|  Electrical|
|      11|      +1 Lap|
|      12|     +2 Laps|
|      13|     +3 Laps|
|      14|     +4 Laps|
|      15|     +5 Laps|
|      16|     +6 Laps|
|      17|     +7 Laps|
|      18|     +8 Laps|
|      19|     +9 Laps|
|      20|    Spun off|
+--------+------------+
only showing top 20 rows

root
 |-- statusId: integer (nullable = true)
 |-- status: string (nullable = true)

Row count:  139
Columnns:  ['statusId', 'status']


In [20]:
%%time
# Load status.csv file using a schema with %%time

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# start = time.time()

schema = StructType([
    StructField("statusId", IntegerType(), True),
    StructField("status", StringType(), True)
])

df = spark.read.csv("f1-sourcefiles/status.csv", header=True, schema=schema)

df.show()

# end = time.time()

# print(f"Time takes: {end - start:.2f} seconds")

df.printSchema()  # check column types
print("Row count: ", df.count())       # number of rows
print("Columnns: ", df.columns)       # list of column names

+--------+------------+
|statusId|      status|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
|       6|     Gearbox|
|       7|Transmission|
|       8|      Clutch|
|       9|  Hydraulics|
|      10|  Electrical|
|      11|      +1 Lap|
|      12|     +2 Laps|
|      13|     +3 Laps|
|      14|     +4 Laps|
|      15|     +5 Laps|
|      16|     +6 Laps|
|      17|     +7 Laps|
|      18|     +8 Laps|
|      19|     +9 Laps|
|      20|    Spun off|
+--------+------------+
only showing top 20 rows

root
 |-- statusId: integer (nullable = true)
 |-- status: string (nullable = true)

Row count:  139
Columnns:  ['statusId', 'status']
CPU times: user 7.7 ms, sys: 0 ns, total: 7.7 ms
Wall time: 137 ms


In [30]:
# select() - basic DataFrame operation

from pyspark.sql.functions import col

##  This is like a SELECT statement in SQL 

df.select("statusId").show()

##  This select all columns

df.select("*").show()

## Columns can be selected using a function called col() and alias() is used to rename a column name

df.select(col("statusId"), (col("statusId") + 10).alias("statusid-plus-10")).show()

## Using col() we can also use cast() to change the type

df.select(col("statusId"), (col("statusId") + 10).cast("double").alias("statusid-plus-10-cast-as-double")).show()


## Rule of thumb: If you just need to pick a column as-is, a plain string is fine. The moment you need to transform, rename, or calculate — use col().


+--------+
|statusId|
+--------+
|       1|
|       2|
|       3|
|       4|
|       5|
|       6|
|       7|
|       8|
|       9|
|      10|
|      11|
|      12|
|      13|
|      14|
|      15|
|      16|
|      17|
|      18|
|      19|
|      20|
+--------+
only showing top 20 rows

+--------+------------+
|statusId|      status|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
|       6|     Gearbox|
|       7|Transmission|
|       8|      Clutch|
|       9|  Hydraulics|
|      10|  Electrical|
|      11|      +1 Lap|
|      12|     +2 Laps|
|      13|     +3 Laps|
|      14|     +4 Laps|
|      15|     +5 Laps|
|      16|     +6 Laps|
|      17|     +7 Laps|
|      18|     +8 Laps|
|      19|     +9 Laps|
|      20|    Spun off|
+--------+------------+
only showing top 20 rows

+--------+----------------+
|statusId|statusid-plus-10|
+--------+----------------+
|       1|              

In [4]:
# filter() - basic operations on dataframes using col()

from pyspark.sql.functions import col

df.filter(col("statusId") < 13).show()


+--------+------------+
|statusId|      status|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
|       6|     Gearbox|
|       7|Transmission|
|       8|      Clutch|
|       9|  Hydraulics|
|      10|  Electrical|
|      11|      +1 Lap|
|      12|     +2 Laps|
+--------+------------+



In [6]:
# filter() - basic operations on dataframes using SQL string expression

df.filter("statusId < 7").show()

+--------+------------+
|statusId|      status|
+--------+------------+
|       1|    Finished|
|       2|Disqualified|
|       3|    Accident|
|       4|   Collision|
|       5|      Engine|
|       6|     Gearbox|
+--------+------------+



In [7]:
# filter() - basic operations on dataframes using SQL string expression and exact match

df.filter("status == 'Engine'").show()



+--------+------+
|statusId|status|
+--------+------+
|       5|Engine|
+--------+------+



In [8]:
# filter() - basic operations on dataframes using col() and exact match

df.filter(col("status") == "Engine").show()



+--------+------+
|statusId|status|
+--------+------+
|       5|Engine|
+--------+------+



In [ ]:
# limit() basic operations on dataframes